# Phase 4 Part A — Revenue Adequacy and the Breaking Point

This notebook examines whether ERCOT energy-only market revenue signals are weakening as renewable penetration grows. It uses load-shape scarcity and curtailment-pressure proxies from the existing project data, not direct ERCOT price or curtailment settlement data.

In [1]:

import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
from pathlib import Path
import subprocess
import sys
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

conn = duckdb.connect(str(PROJECT_ROOT / 'ercot_intelligence.db'))

CHROME_PATH = PROJECT_ROOT / '.kaleido_chrome/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing'
CHART_DIR = PROJECT_ROOT / 'outputs/charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)

# Approved export workaround: save Plotly HTML first, then generate PNG from the same HTML via headless Chrome.
def save_chart(fig, filename, width=1200, height=700):
    html_path = CHART_DIR / f'{filename}.html'
    png_path = CHART_DIR / f'{filename}.png'
    fig.write_html(html_path)
    print(f'Chart saved: {html_path.relative_to(PROJECT_ROOT)}')
    subprocess.run([
        str(CHROME_PATH),
        '--headless=new',
        '--disable-gpu',
        '--no-sandbox',
        '--disable-dev-shm-usage',
        f'--screenshot={png_path}',
        f'--window-size={width},{height}',
        f'file://{html_path}',
    ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print(f'Chart saved: {png_path.relative_to(PROJECT_ROOT)}')

# Load all processed data
gen_mix = pd.read_csv(PROJECT_ROOT / 'data/processed/gen_mix_analysis.csv')
renewable_share = pd.read_csv(PROJECT_ROOT / 'data/processed/renewable_share.csv')
load_data = pd.read_csv(PROJECT_ROOT / 'data/processed/load_clean.csv', parse_dates=['datetime'])
price_data = pd.read_csv(PROJECT_ROOT / 'data/processed/price_analysis.csv')

print('Data loaded successfully')
print(f'  Generation mix: {gen_mix.shape}')
print(f'  Renewable share: {renewable_share.shape}')
print(f'  Load data: {load_data.shape}')
print(f'  Price data: {price_data.shape}')
print(f'  Chrome export path exists: {CHROME_PATH.exists()}')

# Verify date ranges
print(f"\n  Load data range: {load_data['datetime'].min()} to {load_data['datetime'].max()}")
print(f"  Renewable share range: {renewable_share['year'].min()} to {renewable_share['year'].max()}")
print(f"  Load zones: {sorted(load_data['zone'].unique())}")
print(f"  Renewable columns: {renewable_share.columns.tolist()}")


Data loaded successfully
  Generation mix: (840, 10)
  Renewable share: (120, 14)
  Load data: (789039, 7)
  Price data: (120, 20)
  Chrome export path exists: True

  Load data range: 2015-01-01 01:00:00 to 2024-12-31 23:00:00
  Renewable share range: 2015 to 2024
  Load zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']
  Renewable columns: ['year', 'month', 'month_date', 'total_generation_mwh', 'renewable_mwh', 'wind_mwh', 'solar_mwh', 'gas_mwh', 'coal_mwh', 'renewable_pct', 'solar_pct', 'wind_pct', 'renewable_pct_prior_year', 'renewable_yoy_change']


In [2]:

print('ANALYSIS A: SCARCITY HOUR FREQUENCY')
print('=' * 50)

annual_peak = load_data[load_data['zone'] == 'TOTAL'].groupby('year')['load_mw'].max().reset_index()
annual_peak.columns = ['year', 'annual_peak_mw']
print('\nAnnual peak load:')
print(annual_peak.to_string(index=False))

load_total = load_data[load_data['zone'] == 'TOTAL'].copy()
load_total = load_total.merge(annual_peak, on='year')
load_total['scarcity_threshold_mw'] = load_total['annual_peak_mw'] * 0.90
load_total['is_scarcity_hour'] = (load_total['load_mw'] >= load_total['scarcity_threshold_mw']).astype(int)
print('\nLoad-total sample after scarcity flag:')
print(load_total[['datetime', 'year', 'load_mw', 'annual_peak_mw', 'scarcity_threshold_mw', 'is_scarcity_hour']].head(10).to_string(index=False))

scarcity_by_year = load_total.groupby('year').agg(
    total_hours=('load_mw', 'count'),
    scarcity_hours=('is_scarcity_hour', 'sum'),
    annual_peak_mw=('annual_peak_mw', 'first'),
    avg_load_mw=('load_mw', 'mean'),
).reset_index()
scarcity_by_year['scarcity_hours_pct'] = (scarcity_by_year['scarcity_hours'] / scarcity_by_year['total_hours'] * 100).round(2)

scarcity_merged = scarcity_by_year.merge(
    renewable_share.groupby('year')['renewable_pct'].mean().reset_index(),
    on='year'
)

print('\nScarcity hours by year (hours where load >= 90% of annual peak):')
print(scarcity_merged[['year', 'scarcity_hours', 'scarcity_hours_pct', 'annual_peak_mw', 'renewable_pct']].to_string(index=False))

X = sm.add_constant(scarcity_merged['renewable_pct'])
y = scarcity_merged['scarcity_hours']
model_scarcity = sm.OLS(y, X).fit(cov_type='HC1')

print('\nOLS: Scarcity Hours ~ Renewable Share')
print(f"  Coefficient on renewable_pct: {model_scarcity.params['renewable_pct']:.1f} hours per ppt")
print(f"  R-squared: {model_scarcity.rsquared:.3f}")
print(f"  P-value: {model_scarcity.pvalues['renewable_pct']:.4f}")
print('  Interpretation: Each 1 ppt increase in renewable share is associated with')
print(f"  {abs(model_scarcity.params['renewable_pct']):.0f} {'fewer' if model_scarcity.params['renewable_pct'] < 0 else 'more'} scarcity hours per year")


ANALYSIS A: SCARCITY HOUR FREQUENCY

Annual peak load:
 year  annual_peak_mw
 2015    69620.407614
 2016    71092.609221
 2017    69496.239761
 2018    73308.153447
 2019    74665.579486
 2020    74327.836839
 2021    73650.573480
 2022    80037.836007
 2023    85464.116394
 2024    85198.850050

Load-total sample after scarcity flag:
           datetime  year      load_mw  annual_peak_mw  scarcity_threshold_mw  is_scarcity_hour
2015-01-01 01:00:00  2015 39624.861027    69620.407614           62658.366853                 0
2015-01-01 02:00:00  2015 39013.544802    69620.407614           62658.366853                 0
2015-01-01 03:00:00  2015 38566.541927    69620.407614           62658.366853                 0
2015-01-01 04:00:00  2015 38488.338511    69620.407614           62658.366853                 0
2015-01-01 05:00:00  2015 38812.835564    69620.407614           62658.366853                 0
2015-01-01 06:00:00  2015 39660.455969    69620.407614           62658.366853          

In [3]:

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=scarcity_merged['year'], y=scarcity_merged['scarcity_hours'], name='Scarcity Hours (load >= 90% peak)', marker_color='#E53935', opacity=0.8), secondary_y=False)
fig.add_trace(go.Scatter(x=scarcity_merged['year'], y=scarcity_merged['renewable_pct'], name='Renewable Share (%)', line=dict(color='#4CAF50', width=3), mode='lines+markers', marker=dict(size=8)), secondary_y=True)
trend_x = scarcity_merged['year']
trend_y = model_scarcity.params['const'] + model_scarcity.params['renewable_pct'] * scarcity_merged['renewable_pct']
fig.add_trace(go.Scatter(x=trend_x, y=trend_y, name='Scarcity trend', line=dict(color='#B71C1C', width=2, dash='dash'), mode='lines'), secondary_y=False)
fig.update_layout(title=dict(text='ERCOT Scarcity Hours vs Renewable Share<br><sub>Scarcity proxy: hours where load exceeds 90% of annual peak</sub>', font=dict(size=16)), xaxis_title='Year', plot_bgcolor='white', paper_bgcolor='white', hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.05), barmode='overlay', margin=dict(l=70, r=70, t=90, b=80))
fig.update_yaxes(title_text='Scarcity Hours per Year', secondary_y=False, color='#E53935', showgrid=True, gridcolor='#F5F5F5')
fig.update_yaxes(title_text='Renewable Share (%)', secondary_y=True, color='#4CAF50', ticksuffix='%')
fig.add_annotation(xref='paper', yref='paper', x=1.0, y=-0.12, text='Source: ERCOT Hourly Load Data; EIA Texas Generation Data', showarrow=False, font=dict(size=10, color='gray'), xanchor='right')
save_chart(fig, 'chart_7a_scarcity_hours_declining')
fig.show()

latest_scarcity = scarcity_merged[scarcity_merged['year'] == scarcity_merged['year'].max()]['scarcity_hours'].values[0]
earliest_scarcity = scarcity_merged[scarcity_merged['year'] == scarcity_merged['year'].min()]['scarcity_hours'].values[0]
print('\n' + '=' * 50)
print('KEY FINDING A:')
print(f"  Scarcity hours in {scarcity_merged['year'].min()}: {earliest_scarcity:.0f} hours")
print(f"  Scarcity hours in {scarcity_merged['year'].max()}: {latest_scarcity:.0f} hours")
print(f"  Change: {latest_scarcity - earliest_scarcity:.0f} hours ({(latest_scarcity / earliest_scarcity - 1) * 100:.0f}%)")
print(f"  Each 1 ppt increase in renewable share: {model_scarcity.params['renewable_pct']:.1f} scarcity hours")
print('=' * 50)


Chart saved: outputs/charts/chart_7a_scarcity_hours_declining.html


Chart saved: outputs/charts/chart_7a_scarcity_hours_declining.png



KEY FINDING A:
  Scarcity hours in 2015: 176 hours
  Scarcity hours in 2024: 279 hours
  Change: 103 hours (59%)
  Each 1 ppt increase in renewable share: 9.2 scarcity hours


In [4]:

print('ANALYSIS B: IMPLIED CURTAILMENT PRESSURE')
print('=' * 50)

load_total_b = load_data[load_data['zone'] == 'TOTAL'].copy()
p10_by_year = load_total_b.groupby('year')['load_mw'].quantile(0.10).reset_index()
p10_by_year.columns = ['year', 'p10_load_mw']
print('\nAnnual 10th percentile load thresholds:')
print(p10_by_year.to_string(index=False))

load_total_b = load_total_b.merge(p10_by_year, on='year')
load_total_b['is_low_demand'] = (load_total_b['load_mw'] <= load_total_b['p10_load_mw']).astype(int)
load_total_b['is_spring'] = load_total_b['month'].isin([3, 4, 5]).astype(int)
load_total_b['is_curtailment_risk'] = ((load_total_b['is_low_demand'] == 1) & (load_total_b['is_spring'] == 1)).astype(int)
print('\nCurtailment-risk flag sample:')
print(load_total_b[['datetime', 'year', 'month', 'load_mw', 'p10_load_mw', 'is_low_demand', 'is_spring', 'is_curtailment_risk']].head(10).to_string(index=False))

curtailment_pressure = load_total_b.groupby('year').agg(
    curtailment_risk_hours=('is_curtailment_risk', 'sum'),
    total_spring_hours=('is_spring', 'sum'),
).reset_index()
curtailment_pressure['curtailment_risk_pct'] = (curtailment_pressure['curtailment_risk_hours'] / curtailment_pressure['total_spring_hours'] * 100).round(2)

curtailment_merged = curtailment_pressure.merge(
    renewable_share.groupby('year')['renewable_pct'].mean().reset_index(),
    on='year'
)

print('\nCurtailment pressure hours by year (spring low-demand hours):')
print(curtailment_merged[['year', 'curtailment_risk_hours', 'curtailment_risk_pct', 'renewable_pct']].to_string(index=False))

X_c = sm.add_constant(curtailment_merged['renewable_pct'])
y_c = curtailment_merged['curtailment_risk_hours']
model_curtailment = sm.OLS(y_c, X_c).fit(cov_type='HC1')

print('\nOLS: Curtailment Pressure Hours ~ Renewable Share')
print(f"  Coefficient: {model_curtailment.params['renewable_pct']:.1f} hours per ppt")
print(f"  R-squared: {model_curtailment.rsquared:.3f}")
print(f"  P-value: {model_curtailment.pvalues['renewable_pct']:.4f}")


ANALYSIS B: IMPLIED CURTAILMENT PRESSURE

Annual 10th percentile load thresholds:
 year  p10_load_mw
 2015 29343.704327
 2016 29474.780255
 2017 30348.472297
 2018 32298.677448
 2019 33473.725052
 2020 33364.204496
 2021 34530.970757
 2022 37799.274523
 2023 38831.484955
 2024 41353.551523

Curtailment-risk flag sample:
           datetime  year  month      load_mw  p10_load_mw  is_low_demand  is_spring  is_curtailment_risk
2015-01-01 01:00:00  2015      1 39624.861027 29343.704327              0          0                    0
2015-01-01 02:00:00  2015      1 39013.544802 29343.704327              0          0                    0
2015-01-01 03:00:00  2015      1 38566.541927 29343.704327              0          0                    0
2015-01-01 04:00:00  2015      1 38488.338511 29343.704327              0          0                    0
2015-01-01 05:00:00  2015      1 38812.835564 29343.704327              0          0                    0
2015-01-01 06:00:00  2015      1 39660.455

In [5]:

fig_b = make_subplots(specs=[[{'secondary_y': True}]])
fig_b.add_trace(go.Bar(x=curtailment_merged['year'], y=curtailment_merged['curtailment_risk_hours'], name='Curtailment Pressure Hours', marker_color='#FF8F00', opacity=0.85), secondary_y=False)
fig_b.add_trace(go.Scatter(x=curtailment_merged['year'], y=curtailment_merged['renewable_pct'], name='Renewable Share (%)', line=dict(color='#4CAF50', width=3), mode='lines+markers', marker=dict(size=8)), secondary_y=True)
fig_b.update_layout(title=dict(text='ERCOT Curtailment Pressure vs Renewable Penetration<br><sub>Spring low-demand hours where renewable oversupply risk is highest</sub>', font=dict(size=16)), xaxis_title='Year', plot_bgcolor='white', paper_bgcolor='white', hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.05), margin=dict(l=70, r=70, t=90, b=80))
fig_b.update_yaxes(title_text='Curtailment Pressure Hours', secondary_y=False, color='#FF8F00', showgrid=True, gridcolor='#F5F5F5')
fig_b.update_yaxes(title_text='Renewable Share (%)', secondary_y=True, color='#4CAF50', ticksuffix='%')
fig_b.add_annotation(xref='paper', yref='paper', x=1.0, y=-0.12, text='Source: ERCOT Hourly Load Data; EIA Texas Generation Data', showarrow=False, font=dict(size=10, color='gray'), xanchor='right')
save_chart(fig_b, 'chart_7b_curtailment_pressure_growing')
fig_b.show()

early_curtailment = curtailment_merged[curtailment_merged['year'] == 2015]['curtailment_risk_hours'].values[0]
late_curtailment = curtailment_merged[curtailment_merged['year'] == curtailment_merged['year'].max()]['curtailment_risk_hours'].values[0]
print('\n' + '=' * 50)
print('KEY FINDING B:')
print(f'  Curtailment pressure hours in 2015: {early_curtailment:.0f} hours')
print(f"  Curtailment pressure hours in {curtailment_merged['year'].max()}: {late_curtailment:.0f} hours")
print(f"  Increase: {late_curtailment - early_curtailment:.0f} hours ({(late_curtailment / early_curtailment - 1) * 100:.0f}%)")
print('  Note: external ERCOT reporting has cited multi-TWh renewable curtailment in recent years; this notebook uses load-shape pressure as a proxy, not direct curtailment metering.')
print('=' * 50)


Chart saved: outputs/charts/chart_7b_curtailment_pressure_growing.html


Chart saved: outputs/charts/chart_7b_curtailment_pressure_growing.png



KEY FINDING B:
  Curtailment pressure hours in 2015: 393 hours
  Curtailment pressure hours in 2024: 433 hours
  Increase: 40 hours (10%)
  Note: external ERCOT reporting has cited multi-TWh renewable curtailment in recent years; this notebook uses load-shape pressure as a proxy, not direct curtailment metering.


In [6]:

print('ANALYSIS C: REVENUE ADEQUACY INDEX')
print('=' * 50)

ra_df = scarcity_merged.merge(curtailment_merged[['year', 'curtailment_risk_hours', 'curtailment_risk_pct']], on='year')
baseline = ra_df[ra_df['year'].isin([2015, 2016, 2017])]
baseline_scarcity = baseline['scarcity_hours'].mean()
baseline_curtailment = baseline['curtailment_risk_pct'].mean()

print('Baseline period (2015-2017):')
print(f'  Average scarcity hours: {baseline_scarcity:.0f}')
print(f'  Average curtailment pressure: {baseline_curtailment:.1f}%')
print('\nBaseline rows:')
print(baseline[['year', 'scarcity_hours', 'curtailment_risk_pct', 'renewable_pct']].to_string(index=False))

ra_df['scarcity_index'] = ra_df['scarcity_hours'] / baseline_scarcity
ra_df['curtailment_index'] = ra_df['curtailment_risk_pct'] / baseline_curtailment
ra_df['revenue_adequacy_index'] = (ra_df['scarcity_index'] - ra_df['curtailment_index']).round(3)

print('\nRevenue Adequacy Index by year:')
print('(Higher = better price signals for investment)')
print(ra_df[['year', 'scarcity_hours', 'curtailment_risk_pct', 'renewable_pct', 'revenue_adequacy_index']].to_string(index=False))

positive_years = ra_df[ra_df['revenue_adequacy_index'] > 0]
negative_years = ra_df[ra_df['revenue_adequacy_index'] < 0]
if len(negative_years) > 0:
    inflection_year = int(negative_years['year'].min())
    inflection_renewable = ra_df[ra_df['year'] == inflection_year]['renewable_pct'].values[0]
    print('\nINFLECTION POINT:')
    print(f'  RAI turned negative in: {inflection_year}')
    print(f'  Renewable share at that point: {inflection_renewable:.1f}%')
    print('  This is the first year the composite market signal turned negative under this proxy index')
else:
    current_rai = ra_df['revenue_adequacy_index'].iloc[-1]
    current_renewable = ra_df['renewable_pct'].iloc[-1]
    print('\n  RAI still positive as of latest year')
    print(f'  Current RAI: {current_rai:.3f}')
    print(f'  Current renewable share: {current_renewable:.1f}%')
    print('  Watch for RAI approaching zero as renewables grow further')


ANALYSIS C: REVENUE ADEQUACY INDEX
Baseline period (2015-2017):
  Average scarcity hours: 196
  Average curtailment pressure: 18.3%

Baseline rows:
 year  scarcity_hours  curtailment_risk_pct  renewable_pct
 2015             176                 17.81      10.299167
 2016             171                 19.26      13.186750
 2017             242                 17.81      15.831917

Revenue Adequacy Index by year:
(Higher = better price signals for investment)
 year  scarcity_hours  curtailment_risk_pct  renewable_pct  revenue_adequacy_index
 2015             176                 17.81      10.299167                  -0.077
 2016             171                 19.26      13.186750                  -0.182
 2017             242                 17.81      15.831917                   0.259
 2018             237                 19.62      16.976833                   0.135
 2019             222                 19.62      18.704333                   0.058
 2020             229                 

In [7]:

fig_c = make_subplots(specs=[[{'secondary_y': True}]])
colors = ['#E53935' if v < 0 else '#1E88E5' for v in ra_df['revenue_adequacy_index']]
fig_c.add_trace(go.Bar(x=ra_df['year'], y=ra_df['revenue_adequacy_index'], name='Revenue Adequacy Index', marker_color=colors, opacity=0.85), secondary_y=False)
fig_c.add_trace(go.Scatter(x=ra_df['year'], y=ra_df['renewable_pct'], name='Renewable Share (%)', line=dict(color='#4CAF50', width=3), mode='lines+markers', marker=dict(size=8)), secondary_y=True)
fig_c.add_hline(y=0, line_dash='solid', line_color='black', line_width=1.5, secondary_y=False)
fig_c.add_annotation(x=ra_df['year'].mean(), y=0.05, text='Above zero: stronger scarcity signal relative to curtailment pressure', showarrow=False, font=dict(size=11, color='#1E88E5'))
fig_c.add_annotation(x=ra_df['year'].mean(), y=-0.15, text='Below zero: weakening investment signals', showarrow=False, font=dict(size=11, color='#E53935'))
fig_c.update_layout(title=dict(text='ERCOT Revenue Adequacy Index 2015-2024<br><sub>Composite of scarcity hours vs curtailment pressure</sub>', font=dict(size=16)), xaxis_title='Year', plot_bgcolor='white', paper_bgcolor='white', hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.05), margin=dict(l=70, r=70, t=90, b=80))
fig_c.update_yaxes(title_text='Revenue Adequacy Index (normalized)', secondary_y=False, showgrid=True, gridcolor='#F5F5F5')
fig_c.update_yaxes(title_text='Renewable Share (%)', secondary_y=True, color='#4CAF50', ticksuffix='%')
fig_c.add_annotation(xref='paper', yref='paper', x=1.0, y=-0.12, text='Source: ERCOT Hourly Load Data; EIA Texas Generation Data', showarrow=False, font=dict(size=10, color='gray'), xanchor='right')
save_chart(fig_c, 'chart_7c_revenue_adequacy_index')
fig_c.show()


Chart saved: outputs/charts/chart_7c_revenue_adequacy_index.html


Chart saved: outputs/charts/chart_7c_revenue_adequacy_index.png


In [8]:

print('SYNTHESIS: THE BREAKING POINT TRAJECTORY')
print('=' * 50)

synthesis = ra_df[['year', 'renewable_pct', 'revenue_adequacy_index', 'scarcity_hours', 'curtailment_risk_pct']].copy()
print('\nHistorical synthesis table:')
print(synthesis.to_string(index=False))

try:
    forecast = pd.read_csv(PROJECT_ROOT / 'outputs/tableau_exports/renewable_forecast.csv', parse_dates=['date'])
    forecast['year'] = forecast['date'].dt.year
    forecast_annual = forecast.groupby('year')['forecast_pct'].mean().reset_index()
    forecast_annual = forecast_annual[forecast_annual['year'] > synthesis['year'].max()]
    print(f'\n  Loaded renewable forecast for {len(forecast_annual)} future years')
    print(forecast_annual.to_string(index=False))
except Exception as e:
    print(f'  Warning: could not load forecast — {e}')
    forecast_annual = None

if forecast_annual is not None and len(forecast_annual) > 0:
    X_rai = sm.add_constant(ra_df['renewable_pct'])
    y_rai = ra_df['revenue_adequacy_index']
    model_rai = sm.OLS(y_rai, X_rai).fit()
    forecast_annual['projected_rai'] = model_rai.params['const'] + model_rai.params['renewable_pct'] * forecast_annual['forecast_pct']
    print('\nRAI projection model:')
    print(f"  Intercept: {model_rai.params['const']:.3f}")
    print(f"  Renewable coefficient: {model_rai.params['renewable_pct']:.3f} RAI units per ppt")
    print(f"  R-squared: {model_rai.rsquared:.3f}")
    print('\nProjected annual RAI:')
    print(forecast_annual.to_string(index=False))
    threshold_df = forecast_annual[forecast_annual['projected_rai'] <= -0.5]
    if len(threshold_df) > 0:
        concern_year = int(threshold_df['year'].min())
        concern_renewable = threshold_df[threshold_df['year'] == concern_year]['forecast_pct'].values[0]
        print(f'\n  Projected year when RAI reaches -0.5 threshold: {concern_year}')
        print(f'  Renewable share at that point: {concern_renewable:.1f}%')
        print('  This represents a significant weakening of market adequacy signals')
    else:
        print('  RAI does not reach -0.5 threshold within forecast period')
        print(f"  Projected RAI in 2030: {forecast_annual['projected_rai'].iloc[-1]:.3f}")

print('\nFINAL SYNTHESIS:')
print('  Three measures describe the market signal trajectory:')
print('  1. Scarcity hours: proxy for revenue opportunities for peakers')
print('  2. Curtailment pressure: proxy for renewable revenue erosion risk')
print('  3. Revenue Adequacy Index: composite indicator of market signal strength')
print('  This does not prove ERCOT is broken today; it flags a trajectory investors should monitor as renewables grow.')


SYNTHESIS: THE BREAKING POINT TRAJECTORY

Historical synthesis table:
 year  renewable_pct  revenue_adequacy_index  scarcity_hours  curtailment_risk_pct
 2015      10.299167                  -0.077             176                 17.81
 2016      13.186750                  -0.182             171                 19.26
 2017      15.831917                   0.259             242                 17.81
 2018      16.976833                   0.135             237                 19.62
 2019      18.704333                   0.058             222                 19.62
 2020      21.880750                   0.020             229                 20.98
 2021      24.660833                   0.561             330                 20.48
 2022      26.959167                   0.804             343                 17.26
 2023      28.216583                   1.032             400                 18.40
 2024      30.212667                   0.349             279                 19.62

  Loaded renewab

In [9]:

fig_synth = make_subplots(rows=3, cols=1, subplot_titles=['Scarcity Hours (hours >= 90% annual peak)', 'Curtailment Pressure (spring low-demand hours)', 'Revenue Adequacy Index (composite)'], vertical_spacing=0.12, specs=[[{'secondary_y': True}], [{'secondary_y': True}], [{'secondary_y': True}]])
fig_synth.add_trace(go.Bar(x=scarcity_merged['year'], y=scarcity_merged['scarcity_hours'], name='Scarcity Hours', marker_color='#E53935', showlegend=True), row=1, col=1, secondary_y=False)
fig_synth.add_trace(go.Scatter(x=scarcity_merged['year'], y=scarcity_merged['renewable_pct'], name='Renewable %', line=dict(color='#4CAF50', width=2), mode='lines+markers', showlegend=True), row=1, col=1, secondary_y=True)
fig_synth.add_trace(go.Bar(x=curtailment_merged['year'], y=curtailment_merged['curtailment_risk_hours'], name='Curtailment Hours', marker_color='#FF8F00', showlegend=True), row=2, col=1, secondary_y=False)
fig_synth.add_trace(go.Scatter(x=curtailment_merged['year'], y=curtailment_merged['renewable_pct'], name='Renewable %', line=dict(color='#4CAF50', width=2), mode='lines+markers', showlegend=False), row=2, col=1, secondary_y=True)
rai_colors = ['#E53935' if v < 0 else '#1E88E5' for v in ra_df['revenue_adequacy_index']]
fig_synth.add_trace(go.Bar(x=ra_df['year'], y=ra_df['revenue_adequacy_index'], name='RAI', marker_color=rai_colors, showlegend=True), row=3, col=1, secondary_y=False)
fig_synth.add_trace(go.Scatter(x=ra_df['year'], y=ra_df['renewable_pct'], name='Renewable %', line=dict(color='#4CAF50', width=2), mode='lines+markers', showlegend=False), row=3, col=1, secondary_y=True)
fig_synth.update_layout(title=dict(text='ERCOT Market Adequacy: Three Revenue-Signal Indicators<br><sub>Scarcity, curtailment pressure, and composite adequacy signals as renewable penetration grows</sub>', font=dict(size=16)), height=1000, plot_bgcolor='white', paper_bgcolor='white', hovermode='x unified', margin=dict(l=70, r=70, t=100, b=80))
fig_synth.add_annotation(xref='paper', yref='paper', x=1.0, y=-0.06, text='Source: ERCOT Hourly Load Data; EIA Texas Generation Data; author calculations', showarrow=False, font=dict(size=10, color='gray'), xanchor='right')
save_chart(fig_synth, 'chart_7d_market_adequacy_synthesis', height=1100)
fig_synth.show()


Chart saved: outputs/charts/chart_7d_market_adequacy_synthesis.html


Chart saved: outputs/charts/chart_7d_market_adequacy_synthesis.png


In [10]:

processed_dir = PROJECT_ROOT / 'data/processed'
tableau_dir = PROJECT_ROOT / 'outputs/tableau_exports'
print('Verifying output directories before writing:')
print(f'  {processed_dir}: {processed_dir.exists()}')
print(f'  {tableau_dir}: {tableau_dir.exists()}')

scarcity_merged.to_csv(processed_dir / 'scarcity_analysis.csv', index=False)
curtailment_merged.to_csv(processed_dir / 'curtailment_analysis.csv', index=False)
ra_df.to_csv(processed_dir / 'revenue_adequacy_index.csv', index=False)
scarcity_merged.to_csv(tableau_dir / 'scarcity_hours.csv', index=False)
ra_df.to_csv(tableau_dir / 'revenue_adequacy_index.csv', index=False)

print('Notebook 07 outputs saved:')
for path in [
    processed_dir / 'scarcity_analysis.csv',
    processed_dir / 'curtailment_analysis.csv',
    processed_dir / 'revenue_adequacy_index.csv',
    tableau_dir / 'scarcity_hours.csv',
    tableau_dir / 'revenue_adequacy_index.csv',
]:
    rows = len(pd.read_csv(path))
    size_kb = path.stat().st_size / 1024
    print(f'  {path.relative_to(PROJECT_ROOT)}: {rows} rows, {size_kb:.1f} KB')

print('\n' + '=' * 60)
print('NOTEBOOK 07 COMPLETE')
print('Key findings:')
print(f"  A. Scarcity hours changed {latest_scarcity - earliest_scarcity:.0f} hours from 2015 to {scarcity_merged['year'].max()}")
print(f"  B. Curtailment pressure hours changed {late_curtailment - early_curtailment:.0f} hours since 2015")
print(f"  C. Revenue Adequacy Index current value: {ra_df['revenue_adequacy_index'].iloc[-1]:.3f}")
print('=' * 60)

conn.close()


Verifying output directories before writing:
  /Users/chokkapusaatvika/Desktop/ERCO_Project/data/processed: True
  /Users/chokkapusaatvika/Desktop/ERCO_Project/outputs/tableau_exports: True
Notebook 07 outputs saved:
  data/processed/scarcity_analysis.csv: 10 rows, 0.8 KB
  data/processed/curtailment_analysis.csv: 10 rows, 0.5 KB
  data/processed/revenue_adequacy_index.csv: 10 rows, 1.4 KB
  outputs/tableau_exports/scarcity_hours.csv: 10 rows, 0.8 KB
  outputs/tableau_exports/revenue_adequacy_index.csv: 10 rows, 1.4 KB

NOTEBOOK 07 COMPLETE
Key findings:
  A. Scarcity hours changed 103 hours from 2015 to 2024
  B. Curtailment pressure hours changed 40 hours since 2015
  C. Revenue Adequacy Index current value: 0.349
